# Process single cell profiles

NOTE: We are normalizing the single-cell profiles to all samples on the plate (MAD robustize).

## Import libraries

In [1]:
import pathlib

import pandas as pd
from pycytominer import annotate, feature_select, normalize


## Set paths and variables

In [2]:
# Set the plate to process
plate_id = "CARD-CelIns-CX7_261022150001"

# Directory with QC-labeled profiles
qc_labeled_dir = pathlib.Path("./data/qc_labeled_profiles/").resolve(strict=True)

# Directory to save single-cell profiles
output_dir = pathlib.Path("./data/single_cell_profiles/")
output_dir.mkdir(parents=True, exist_ok=True)

# Path to the platemap for the pilot plate
platemap_path = pathlib.Path(
    "../metadata/platemaps/pilot_platemap.csv"
).resolve(strict=True)

# Path to the QC-labeled profile for the pilot plate
profile_path = (qc_labeled_dir / f"{plate_id}_qc_labeled.parquet").resolve(
    strict=True
)

# operations to perform for feature selection
feature_select_ops = [
    "blocklist",
    "drop_na_columns",
    "variance_threshold",
    "correlation_threshold",
]


## Process data with pycytominer

In [3]:
print("Performing preprocessing on", plate_id)

output_annotated_file = str(output_dir / f"{plate_id}_sc_annotated.parquet")
output_normalized_file = str(output_dir / f"{plate_id}_sc_normalized.parquet")
output_feature_select_file = str(
    output_dir / f"{plate_id}_sc_feature_selected.parquet"
)

profile_df = pd.read_parquet(profile_path)
platemap_df = pd.read_csv(platemap_path)

# Drop all rows in the profiles that failed any Metadata_cqc columns
cqc_columns = [col for col in profile_df.columns if col.startswith("Metadata_cqc")]
if cqc_columns:
    profile_df = profile_df[~profile_df[cqc_columns].any(axis=1)]

print("Performing annotation for", plate_id, "...")
# Step 1: Annotation
annotate(
    profiles=profile_df,
    platemap=platemap_df,
    join_on=["Metadata_well_position", "Image_Metadata_Well"],
    output_file=output_annotated_file,
    output_type="parquet",
)

# Load the annotated parquet file to fix metadata columns names
annotated_df = pd.read_parquet(output_annotated_file)

# Rename columns
annotated_df.rename(columns={"Image_Metadata_Site": "Metadata_Site"}, inplace=True)

# Save back
annotated_df.to_parquet(output_annotated_file, index=False)

# Step 2: Normalization
normalized_df = normalize(
    profiles=output_annotated_file,
    method="mad_robustize",
    output_file=output_normalized_file,
    output_type="parquet",
    samples="all",
)

# Step 3: Feature selection
print("Performing feature selection for", plate_id, "...")
feature_select(
    profiles=normalized_df,
    operation=feature_select_ops,
    na_cutoff=0,
    output_file=output_feature_select_file,
    output_type="parquet",
    blocklist_file="./blocklist_features.txt",
)

print(f"Annotation, normalization, and feature selection complete for {plate_id}")


Performing preprocessing on CARD-CelIns-CX7_261022150001
Performing annotation for CARD-CelIns-CX7_261022150001 ...
Performing feature selection for CARD-CelIns-CX7_261022150001 ...
Annotation, normalization, and feature selection complete for CARD-CelIns-CX7_261022150001


In [4]:
# Check output file
test_df = pd.read_parquet(output_feature_select_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)


(11736, 1033)
Plate: ['CARD-CelIns-CX7_261022150001']
Metadata columns: ['Metadata_WellRow', 'Metadata_WellCol', 'Metadata_Treatment', 'Metadata_N_Replicate', 'Metadata_Nuclei_Location_Center_X', 'Metadata_Nuclei_Location_Center_Y', 'Metadata_Cells_Location_Center_X', 'Metadata_Cells_Location_Center_Y', 'Metadata_Image_Count_Cells', 'Metadata_ImageNumber', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Cells_Number_Object_Number', 'Metadata_Cytoplasm_Parent_Cells', 'Metadata_Cytoplasm_Parent_Nuclei', 'Metadata_Nuclei_Number_Object_Number', 'Metadata_cqc_failed_oversegmented_nuclei', 'Metadata_cqc_failed_small_cells', 'Metadata_cqc_failed_low_intensity', 'Metadata_cqc_failed_blurry_cells', 'Metadata_cqc_failed_missegmented_cells', 'Metadata_Site']


,Metadata_WellRow,Metadata_WellCol,Metadata_Treatment,Metadata_N_Replicate,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,...,Nuclei_Texture_InverseDifferenceMoment_ER_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Golgi_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_SumEntropy_Golgi_3_01_256,Nuclei_Texture_SumEntropy_Mito_3_01_256,Nuclei_Texture_SumVariance_Actin_3_01_256,Nuclei_Texture_SumVariance_DNA_3_01_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Golgi_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256
0,B,3,TGFB,N2,436.220491,216.531456,447.810243,224.585940,8,2,...,0.810278,0.827943,-0.21933,-0.255121,0.421547,2.161214,0.949947,0.001510,-0.263512,1.244137
1,B,3,TGFB,N2,369.889953,727.496963,393.137873,745.939119,4,17,...,-0.853745,-0.287082,-0.67829,1.052370,0.822222,1.150909,3.180670,3.401715,2.387232,2.041169


In [5]:
# Check output file
test_df = pd.read_parquet(output_annotated_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)


(11736, 2495)
Plate: ['CARD-CelIns-CX7_261022150001']
Metadata columns: ['Metadata_WellRow', 'Metadata_WellCol', 'Metadata_Treatment', 'Metadata_N_Replicate', 'Metadata_Nuclei_Location_Center_X', 'Metadata_Nuclei_Location_Center_Y', 'Metadata_Cells_Location_Center_X', 'Metadata_Cells_Location_Center_Y', 'Metadata_Image_Count_Cells', 'Metadata_ImageNumber', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Cells_Number_Object_Number', 'Metadata_Cytoplasm_Parent_Cells', 'Metadata_Cytoplasm_Parent_Nuclei', 'Metadata_Nuclei_Number_Object_Number', 'Metadata_cqc_failed_oversegmented_nuclei', 'Metadata_cqc_failed_small_cells', 'Metadata_cqc_failed_low_intensity', 'Metadata_cqc_failed_blurry_cells', 'Metadata_cqc_failed_missegmented_cells', 'Metadata_Site']


,Metadata_WellRow,Metadata_WellCol,Metadata_Treatment,Metadata_N_Replicate,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,...,Nuclei_Texture_Variance_ER_3_02_256,Nuclei_Texture_Variance_ER_3_03_256,Nuclei_Texture_Variance_Golgi_3_00_256,Nuclei_Texture_Variance_Golgi_3_01_256,Nuclei_Texture_Variance_Golgi_3_02_256,Nuclei_Texture_Variance_Golgi_3_03_256,Nuclei_Texture_Variance_Mito_3_00_256,Nuclei_Texture_Variance_Mito_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256,Nuclei_Texture_Variance_Mito_3_03_256
0,B,3,TGFB,N2,436.220491,216.531456,447.810243,224.585940,8,2,...,4.403614,3.483990,0.610854,0.595196,0.623505,0.601561,2.292144,2.423379,2.934356,2.276889
1,B,3,TGFB,N2,369.889953,727.496963,393.137873,745.939119,4,17,...,13.782462,13.014404,2.076064,2.102842,2.103051,2.116677,4.056737,3.861279,3.970046,3.704805
